# Exercise 1.1 — Word Embeddings: words as numbers
**Created by Shivank Agarwal**

A computer cannot read the word *king*. It can only work with numbers.
An **embedding** is a list of numbers that stands for a word. Words with a similar meaning get similar lists.

In this notebook you will:
1. Load a ready-made embedding model (GloVe, 50 numbers per word)
2. Look at the vector for a word
3. Do "word maths" like **king − man + woman ≈ queen**
4. Measure how close two words are with **cosine similarity** (the same maths every RAG system uses)

> The first run downloads the model (about 66 MB). After that it loads from your computer.

In [2]:
%pip install gensim

   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   - -------------------------------------- 1.0/24.4 MB 9.3 MB/s eta 0:00:03
   ------- -------------------------------- 4.5/24.4 MB 13.6 MB/s eta 0:00:02
   ------------ --------------------------- 7.3/24.4 MB 14.0 MB/s eta 0:00:02
   ----------------- ---------------------- 10.5/24.4 MB 14.1 MB/s eta 0:00:01
   --------------------- ------------------ 13.1/24.4 MB 14.1 MB/s eta 0:00:01
   -------------------------- ------------- 16.0/24.4 MB 13.7 MB/s eta 0:00:01
   ------------------------------- -------- 19.1/24.4 MB 14.0 MB/s eta 0:00:01
   ------------------------------------ --- 22.0/24.4 MB 14.0 MB/s eta 0:00:01
   ---------------------------------------- 24.4/24.4 MB 13.7 MB/s  0:00:01

   ---------------------------------------- 0/2 [smart_open]
   ---------------------------------------- 0/2 [smart_open]
   -------------------- ------------------- 1/2 [gensim]
   -------------------- ------------------- 1/


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Libraries:
#   numpy  -> maths on lists of numbers (vectors)
#   gensim -> gives us ready-made word vectors (GloVe)
import numpy as np
import gensim.downloader as api

# "glove-wiki-gigaword-50" = GloVe vectors trained on Wikipedia + news text.
# Every word in its vocabulary (400,000 words) becomes a list of 50 numbers.
model = api.load("glove-wiki-gigaword-50")
print("Words in vocabulary:", len(model.key_to_index))

[==================================================] 100.0% 66.0/66.0MB downloaded
Words in vocabulary: 400000


## Step 1 — What does a word look like to the computer?

In [4]:
# ----- TODO 1: Get the vector for the word "king" and print its shape and first 5 numbers -----
# Hint: The model works like a dictionary: model["some_word"]
# Hint: A numpy array has a .shape attribute
v_king = model["king"]  # replace None with your code
print("Shape of the 'king' vector:", v_king.shape)  # expect (50,)
print("First 5 numbers:", v_king[:5])

Shape of the 'king' vector: (50,)
First 5 numbers: [ 0.50451   0.68607  -0.59517  -0.022801  0.60046 ]


## Step 2 — Which words live close to "king"?
`most_similar` finds the words whose vectors point in the most similar direction.

In [5]:
for word, score in model.most_similar("king", topn=5):
    print(f"{word:<10} {score:.3f}")

prince     0.824
queen      0.784
ii         0.775
emperor    0.774
son        0.767


## Step 3 — Word maths (analogies)
If *king* is to *man* as ? is to *woman*, then **king − man + woman** should land near **queen**.

⚠️ Gotcha: if you build the vector yourself and search with it, the closest word is often one of the
words you started with (for example *king* itself). `most_similar(positive=[...], negative=[...])`
does the same maths **and** skips the input words. That is why we use it.

In [6]:
# ----- TODO 2: Use word maths to answer "king - man + woman = ?" -----
# Hint: model.most_similar(positive=[words to add], negative=[words to subtract], topn=1)
answer = model.most_similar(positive=["king", "woman"], negative=["man"], topn=1)  # replace None with your code
print("king - man + woman =", answer)

king - man + woman = [('queen', 0.8523604273796082)]


In [12]:
# The same maths done "by hand" with numpy, to see the gotcha for yourself
new_vector = model["king"] - model["man"] + model["woman"]
print("Raw vector search (input words NOT removed):", model.similar_by_vector(new_vector, topn=3))

Raw vector search (input words NOT removed): [('king', 0.8859834671020508), ('queen', 0.8609582185745239), ('daughter', 0.7684512734413147)]


In [9]:
# ----- TODO 3: Capital cities: "paris - france + india = ?" -----
# Hint: Same pattern as TODO 2. Words to add: paris, india. Word to subtract: france
answer_2 = model.most_similar(positive=["paris", "india"], negative=["france"], topn=1)  # replace None with your code
print("paris - france + india =", answer_2)

paris - france + india = [('delhi', 0.8888585567474365)]


## Step 4 — Cosine similarity by hand
Cosine similarity tells us how much two vectors point the same way:

`cosine(a, b) = (a · b) / (|a| × |b|)`

* close to **1** → very similar meaning
* close to **0** → unrelated

You will use this exact formula again in the next exercise to search documents.

In [ ]:
# ----- TODO 4: Write a cosine_similarity(a, b) function using numpy -----
# Hint: np.dot(a, b) gives a · b
# Hint: np.linalg.norm(a) gives |a|
def cosine_similarity(a, b):
    return 0.0  # replace with the formula

pairs = [("king", "queen"), ("cricket", "football"), ("king", "banana")]
for w1, w2 in pairs:
    mine = cosine_similarity(model[w1], model[w2])
    library = model.similarity(w1, w2)  # gensim's own answer, to check yours
    print(f"{w1:>8} vs {w2:<9} -> my answer: {mine:.3f} | gensim: {library:.3f}")

: 